# Patient Risk Prediction System
**Dataset :** Pima Indians Diabetes Database (UCI / Kaggle)
**Author  :** Stellamaries Syombua
**Stack   :** Python · Pandas · NumPy

> **Goal:** Load the dataset, understand its structure, inspect raw values,
> and document data-quality issues before any transformation.

## 1.1 · Import Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.3f}'.format)
print("Libraries loaded ✓")


Libraries loaded ✓


## 1.2 · Load the Dataset

In [2]:

df = pd.read_csv("C:\\Users\\PC\\Downloads\\archive (8)\\diabetes.csv")
print(f"Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(10)


Dataset loaded: 768 rows × 9 columns


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.600,0.627,50,1
1,1,85,66,29,0,26.600,0.351,31,0
2,8,183,64,0,0,23.300,0.672,32,1
3,1,89,66,23,94,28.100,0.167,21,0
4,0,137,40,35,168,43.100,2.288,33,1
5,5,116,74,0,0,25.600,0.201,30,0
6,3,78,50,32,88,31.000,0.248,26,1
7,10,115,0,0,0,35.300,0.134,29,0
8,2,197,70,45,543,30.500,0.158,53,1
9,8,125,96,0,0,0.000,0.232,54,1


## 1.3 · Column Descriptions


In [3]:
col_descriptions = {
    'Pregnancies'            : 'Number of times pregnant',
    'Glucose'                : 'Plasma glucose concentration (2h oral glucose tolerance test)',
    'BloodPressure'          : 'Diastolic blood pressure (mmHg)',
    'SkinThickness'          : 'Triceps skinfold thickness (mm)',
    'Insulin'                : '2-hour serum insulin (mu U/ml)',
    'BMI'                    : 'Body mass index (weight kg / height m²)',
    'DiabetesPedigreeFunction': 'Diabetes pedigree function (genetic risk score)',
    'Age'                    : 'Age in years',
    'Outcome'                : 'Class label — 1 = Diabetes, 0 = No Diabetes',
}

desc_df = pd.DataFrame(
    col_descriptions.items(),
    columns=['Feature', 'Description']
)
print(desc_df.to_string(index=False))


                 Feature                                                   Description
             Pregnancies                                      Number of times pregnant
                 Glucose Plasma glucose concentration (2h oral glucose tolerance test)
           BloodPressure                               Diastolic blood pressure (mmHg)
           SkinThickness                               Triceps skinfold thickness (mm)
                 Insulin                                2-hour serum insulin (mu U/ml)
                     BMI                       Body mass index (weight kg / height m²)
DiabetesPedigreeFunction               Diabetes pedigree function (genetic risk score)
                     Age                                                  Age in years
                 Outcome                   Class label — 1 = Diabetes, 0 = No Diabetes


## 1.4 · Dataset Shape & Data Types

In [4]:
print("Shape:", df.shape)
print()
print("Data Types:")
print(df.dtypes)


Shape: (768, 9)

Data Types:
Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object


## 1.5 · Statistical Summary

In [5]:
df.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.000,3.845,3.370,0.000,1.000,3.000,6.000,17.000
Glucose,768.000,120.895,31.973,0.000,99.000,117.000,140.250,199.000
BloodPressure,768.000,69.105,19.356,0.000,62.000,72.000,80.000,122.000
SkinThickness,768.000,20.536,15.952,0.000,0.000,23.000,32.000,99.000
Insulin,768.000,79.799,115.244,0.000,0.000,30.500,127.250,846.000
BMI,768.000,31.993,7.884,0.000,27.300,32.000,36.600,67.100
DiabetesPedigreeFunction,768.000,0.472,0.331,0.078,0.244,0.372,0.626,2.420
Age,768.000,33.241,11.760,21.000,24.000,29.000,41.000,81.000
Outcome,768.000,0.349,0.477,0.000,0.000,0.000,1.000,1.000


## 1.6 · Class Distribution (Target Variable)

In [6]:
counts = df['Outcome'].value_counts()
pcts   = df['Outcome'].value_counts(normalize=True) * 100

summary = pd.DataFrame({
    'Label'  : ['No Diabetes (0)', 'Diabetes (1)'],
    'Count'  : [counts[0], counts[1]],
    'Percent': [f'{pcts[0]:.1f}%', f'{pcts[1]:.1f}%'],
})
print(summary.to_string(index=False))
print()
print(f"Class imbalance ratio: {counts[0]/counts[1]:.2f}:1")


          Label  Count Percent
No Diabetes (0)    500   65.1%
   Diabetes (1)    268   34.9%

Class imbalance ratio: 1.87:1


## 1.7 · Zero-as-Missing Values Audit

> These zeros represent **missing data** that was encoded as zero.
> They must be handled before modelling.


In [7]:
ZERO_COLS = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

zero_report = []
for col in ZERO_COLS:
    n_zeros = (df[col] == 0).sum()
    pct     = n_zeros / len(df) * 100
    zero_report.append({
        'Feature'        : col,
        'Zero Count'     : n_zeros,
        'Zero %'         : f'{pct:.1f}%',
        'Severity'       : 'HIGH' if pct > 20 else 'MEDIUM' if pct > 5 else 'LOW'
    })

zero_df = pd.DataFrame(zero_report)
print(zero_df.to_string(index=False))
print()
print("Action: Replace zeros in above columns with NaN before imputation.")


      Feature  Zero Count Zero % Severity
      Glucose           5   0.7%      LOW
BloodPressure          35   4.6%      LOW
SkinThickness         227  29.6%     HIGH
      Insulin         374  48.7%     HIGH
          BMI          11   1.4%      LOW

Action: Replace zeros in above columns with NaN before imputation.


## 1.8 · Duplicate Row Check

In [8]:
n_dups = df.duplicated().sum()
print(f"Duplicate rows: {n_dups}")
if n_dups == 0:
    print("✓ No duplicates found — dataset is clean on this front.")


Duplicate rows: 0
✓ No duplicates found — dataset is clean on this front.


## 1.9 · Outlier Detection (IQR Method)

In [9]:
outlier_report = []
for col in df.columns:
    if col == 'Outcome':
        continue
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr     = q3 - q1
    lower   = q1 - 1.5 * iqr
    upper   = q3 + 1.5 * iqr
    n_out   = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_report.append({
        'Feature'   : col,
        'Q1'        : round(q1, 2),
        'Q3'        : round(q3, 2),
        'Lower Fence': round(lower, 2),
        'Upper Fence': round(upper, 2),
        'Outliers'  : n_out,
        'Outlier %' : f'{n_out/len(df)*100:.1f}%'
    })

pd.DataFrame(outlier_report).to_string(index=False)


'                 Feature     Q1      Q3  Lower Fence  Upper Fence  Outliers Outlier %\n             Pregnancies  1.000   6.000       -6.500       13.500         4      0.5%\n                 Glucose 99.000 140.250       37.120      202.120         5      0.7%\n           BloodPressure 62.000  80.000       35.000      107.000        45      5.9%\n           SkinThickness  0.000  32.000      -48.000       80.000         1      0.1%\n                 Insulin  0.000 127.250     -190.880      318.120        34      4.4%\n                     BMI 27.300  36.600       13.350       50.550        19      2.5%\nDiabetesPedigreeFunction  0.240   0.630       -0.330        1.200        29      3.8%\n                     Age 24.000  41.000       -1.500       66.500         9      1.2%'

## ✅ Notebook 1 Summary

| Check | Result |
|---|---|
| Shape | 768 rows × 9 columns |
| Duplicates | 0 |
| Missing (zeros) | Insulin 48.7%, SkinThickness 29.6%, BloodPressure 4.6% |
| Class balance | 65.1% vs 34.9% (mild imbalance) |
| Top outlier concern | Insulin (34 extreme values) |

